# Energy Forecasting — Data Exploration & Merging

This notebook explores the raw consumption, day-ahead price, and weather datasets, generates the Baden-Württemberg holiday table, aligns timestamps across timezones/DST, merges the key variables, and analyzes target statistics and feature correlations.

## 1. Setup & File Paths

In [1]:
from datetime import timedelta, timezone
from pathlib import Path

import holidays
import pandas as pd
from IPython.display import display

# Resolve repository root and Dataset subdirectories (raw / preprocessed)
PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "Dataset").is_dir():
        PROJECT_ROOT = candidate
        break

DATASET_DIR = PROJECT_ROOT / "Dataset"
RAW_DIR = DATASET_DIR / "raw"
PREPROCESSED_DIR = DATASET_DIR / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CONSUMPTION_FILE = RAW_DIR / "Actual_consumption_201701010000_202601010000_Quarterhour.xlsx"
PRICES_FILE = RAW_DIR / "Day-ahead_prices_201701010000_202601010000_Quarterhour.xlsx"
WEATHER_FILE = RAW_DIR / "open-meteo-48.83N9.15E293m.xlsx"
HOLIDAY_FILE = RAW_DIR / "df_holiday.xlsx"
MERGED_FILE = PREPROCESSED_DIR / "merged_dataset.csv"

# Key columns defined in Dataset/README.md
TARGET_COL = "grid load [MWh]"
PRICE_COL = "Germany/Luxembourg [€/MWh]"
WEATHER_COLS = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "precipitation (mm)",
    "shortwave_radiation (W/m²)",
    "wind_speed_10m (km/h)",
    "dew_point_2m (°C)",
    "weather_code (wmo code)",
]

print("Raw directory:         ", RAW_DIR)
print("Preprocessed directory:", PREPROCESSED_DIR)

Raw directory:          C:\Users\Win11\Documents\GitHub\Energy-Forecasting\Dataset\raw
Preprocessed directory: C:\Users\Win11\Documents\GitHub\Energy-Forecasting\Dataset\preprocessed


## 2. Load Raw Datasets

In [2]:
raw_consumption = pd.read_excel(CONSUMPTION_FILE, skiprows=9)
raw_prices = pd.read_excel(PRICES_FILE, skiprows=9)
raw_weather = pd.read_excel(WEATHER_FILE, skiprows=3)

datasets = {
    "Actual Consumption": raw_consumption,
    "Day-Ahead Prices": raw_prices,
    "Open-Meteo Weather": raw_weather,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")

c:\Users\Win11\Documents\GitHub\Energy-Forecasting\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Win11\Documents\GitHub\Energy-Forecasting\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Actual Consumption: 315,552 rows x 6 columns
Day-Ahead Prices: 315,552 rows x 19 columns
Open-Meteo Weather: 313,824 rows x 10 columns


In [3]:
print("Consumption")
display(raw_consumption.head(3))

print("Energy Prices")
display(raw_prices.head(3))

print("Weather Data")
display(raw_weather.head(3))

Consumption


,Start date,End date,grid load [MWh],Grid load incl. hydro pumped storage [MWh],Hydro pumped storage [MWh],Residual load [MWh]
0,"Jan 1, 2017 12:00 AM","Jan 1, 2017 12:15 AM",1544.0,1608.75,64.75,1528.75
1,"Jan 1, 2017 12:15 AM","Jan 1, 2017 12:30 AM",1529.5,1626.5,97,1512.75
2,"Jan 1, 2017 12:30 AM","Jan 1, 2017 12:45 AM",1529.5,1665.5,136,1510.75


Energy Prices


,Start date,End date,Germany/Luxembourg [€/MWh],∅ DE/LU neighbours [€/MWh],Belgium [€/MWh],Denmark 1 [€/MWh],Denmark 2 [€/MWh],France [€/MWh],Netherlands [€/MWh],Norway 2 [€/MWh],Austria [€/MWh],Poland [€/MWh],Sweden 4 [€/MWh],Switzerland [€/MWh],Czech Republic [€/MWh],DE/AT/LU [€/MWh],Northern Italy [€/MWh],Slovenia [€/MWh],Hungary [€/MWh]
0,"Jan 1, 2017 12:00 AM","Jan 1, 2017 12:15 AM",-,-,42.0,20.96,20.96,58.82,42.0,28.4,-,24.03,24.03,57.04,18.5,20.96,53.0,53.0,57.25
1,"Jan 1, 2017 12:15 AM","Jan 1, 2017 12:30 AM",-,-,42.0,20.96,20.96,58.82,42.0,28.4,-,24.03,24.03,57.04,18.5,20.96,53.0,53.0,57.25
2,"Jan 1, 2017 12:30 AM","Jan 1, 2017 12:45 AM",-,-,42.0,20.96,20.96,58.82,42.0,28.4,-,24.03,24.03,57.04,18.5,20.96,53.0,53.0,57.25


Weather Data


,time,temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),direct_radiation (W/m²),sunshine_duration (s),wind_speed_10m (km/h),weather_code (wmo code),dew_point_2m (°C)
0,2017-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data Types & Non-Numeric Placeholder Tokens
Inspect column `dtypes`, count non-numeric tokens (such as `'-'` or `'#NUM!'`), and convert timestamps to `datetime` and value columns to `float64`.

In [4]:
import locale

def parse_timestamp_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_datetime(series, unit="D", origin="1899-12-30").dt.round("min")
    # Use C locale so English month names ('Mar', 'May', 'Oct') and 'AM'/'PM' parse on German Windows
    locale.setlocale(locale.LC_TIME, "C")
    return pd.to_datetime(series, format="%b %d, %Y %I:%M %p", errors="coerce")


timestamp_config = {
    "Actual Consumption": (["Start date", "End date"], "Start date"),
    "Day-Ahead Prices": (["Start date", "End date"], "Start date"),
    "Open-Meteo Weather": (["time"], "time"),
}

coerced_datasets = {}

for name, df in datasets.items():
    ts_cols, _ = timestamp_config[name]
    coerced_df = df.copy()
    type_rows = []

    for col in df.columns:
        raw_dtype = str(df[col].dtype)
        if col in ts_cols:
            parsed_ts = parse_timestamp_series(df[col])
            invalid_count = int(parsed_ts.isna().sum() - df[col].isna().sum())
            coerced_df[col] = parsed_ts
            type_rows.append({
                "column": col,
                "raw_dtype": raw_dtype,
                "expected_type": "datetime",
                "non_numeric_or_invalid": invalid_count,
                "sample_invalid_tokens": "",
            })
        else:
            s = df[col]
            numeric_s = pd.to_numeric(s, errors="coerce")
            non_numeric_mask = s.notna() & numeric_s.isna()
            non_numeric_count = int(non_numeric_mask.sum())
            sample_tokens = (
                s[non_numeric_mask].astype(str).value_counts().head(3).to_dict()
                if non_numeric_count > 0
                else "None"
            )
            coerced_df[col] = numeric_s
            type_rows.append({
                "column": col,
                "raw_dtype": raw_dtype,
                "expected_type": "float64",
                "non_numeric_or_invalid": non_numeric_count,
                "sample_invalid_tokens": str(sample_tokens),
            })

    coerced_datasets[name] = coerced_df
    print(f"=== {name} ===")
    display(pd.DataFrame(type_rows))

=== Actual Consumption ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,Start date,str,datetime,0,
1,End date,str,datetime,0,
2,grid load [MWh],float64,float64,0,None
3,Grid load incl. hydro pumped storage [MWh],object,float64,18,{'-': 18}
4,Hydro pumped storage [MWh],object,float64,18,{'-': 18}
5,Residual load [MWh],object,float64,14,{'-': 14}


=== Day-Ahead Prices ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,Start date,str,datetime,0,
1,End date,str,datetime,0,
2,Germany/Luxembourg [€/MWh],object,float64,61244,{'-': 61244}
3,∅ DE/LU neighbours [€/MWh],object,float64,101088,{'-': 101088}
4,Belgium [€/MWh],float64,float64,0,None
5,Denmark 1 [€/MWh],float64,float64,0,None
6,Denmark 2 [€/MWh],float64,float64,0,None
7,France [€/MWh],float64,float64,0,None
8,Netherlands [€/MWh],float64,float64,0,None
9,Norway 2 [€/MWh],float64,float64,0,None


=== Open-Meteo Weather ===


,column,raw_dtype,expected_type,non_numeric_or_invalid,sample_invalid_tokens
0,time,datetime64[us],datetime,0,
1,temperature_2m (°C),float64,float64,0,None
2,relative_humidity_2m (%),float64,float64,0,None
3,precipitation (mm),float64,float64,0,None
4,shortwave_radiation (W/m²),float64,float64,0,None
5,direct_radiation (W/m²),float64,float64,0,None
6,sunshine_duration (s),float64,float64,0,None
7,wind_speed_10m (km/h),float64,float64,0,None
8,weather_code (wmo code),float64,float64,0,None
9,dew_point_2m (°C),float64,float64,0,None


,dataset,start_ts,end_ts,full_row_duplicates,duplicate_timestamps,oct_dst_fallback_rows,unexpected_duplicate_rows,march_dst_skipped_slots,unexpected_missing_slots
0,Actual Consumption,2017-01-01,2025-12-31 23:45:00,0,36,72,0,36,0
1,Day-Ahead Prices,2017-01-01,2025-12-31 23:45:00,0,36,72,0,36,0
2,Open-Meteo Weather,2017-01-01,2025-12-13 23:45:00,0,0,0,0,0,0


## 4. Missing Data Analysis

In [6]:
for name, coerced_df in coerced_datasets.items():
    raw_df = datasets[name]
    _, primary_ts = timestamp_config[name]
    ts = coerced_df[primary_ts]
    total_rows = len(coerced_df)

    missing_rows = []
    for col in coerced_df.columns:
        raw_na = int(raw_df[col].isna().sum())
        coerced_na = int(coerced_df[col].isna().sum())
        valid_mask = coerced_df[col].notna()

        missing_rows.append({
            "column": col,
            "raw_nulls": raw_na,
            "placeholder_nulls": coerced_na - raw_na,
            "total_missing": coerced_na,
            "missing_%": round((coerced_na / total_rows) * 100.0, 2),
            "first_valid_ts": ts[valid_mask].min() if valid_mask.any() else None,
            "last_valid_ts": ts[valid_mask].max() if valid_mask.any() else None,
        })

    print(f"=== {name} ===")
    display(pd.DataFrame(missing_rows))

=== Actual Consumption ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,Start date,0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
1,End date,0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
2,grid load [MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
3,Grid load incl. hydro pumped storage [MWh],0,18,18,0.01,2017-01-01,2025-12-31 23:45:00
4,Hydro pumped storage [MWh],0,18,18,0.01,2017-01-01,2025-12-31 23:45:00
5,Residual load [MWh],0,14,14,0.00,2017-01-01,2025-12-31 23:45:00


=== Day-Ahead Prices ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,Start date,0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
1,End date,0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
2,Germany/Luxembourg [€/MWh],0,61244,61244,19.41,2018-10-01,2025-12-31 23:45:00
3,∅ DE/LU neighbours [€/MWh],0,101088,101088,32.04,2019-11-20,2025-12-31 23:45:00
4,Belgium [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
5,Denmark 1 [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
6,Denmark 2 [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
7,France [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
8,Netherlands [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00
9,Norway 2 [€/MWh],0,0,0,0.00,2017-01-01,2025-12-31 23:45:00


=== Open-Meteo Weather ===


,column,raw_nulls,placeholder_nulls,total_missing,missing_%,first_valid_ts,last_valid_ts
0,time,0,0,0,0.0,2017-01-01 00:00:00,2025-12-13 23:45:00
1,temperature_2m (°C),8,0,8,0.0,2017-01-01 02:00:00,2025-12-13 23:45:00
2,relative_humidity_2m (%),8,0,8,0.0,2017-01-01 02:00:00,2025-12-13 23:45:00
3,precipitation (mm),5,0,5,0.0,2017-01-01 01:15:00,2025-12-13 23:45:00
4,shortwave_radiation (W/m²),7,0,7,0.0,2017-01-01 01:45:00,2025-12-13 23:45:00
5,direct_radiation (W/m²),7,0,7,0.0,2017-01-01 01:45:00,2025-12-13 23:45:00
6,sunshine_duration (s),7,0,7,0.0,2017-01-01 01:45:00,2025-12-13 23:45:00
7,wind_speed_10m (km/h),8,0,8,0.0,2017-01-01 02:00:00,2025-12-13 23:45:00
8,weather_code (wmo code),5,0,5,0.0,2017-01-01 01:15:00,2025-12-13 23:45:00
9,dew_point_2m (°C),8,0,8,0.0,2017-01-01 02:00:00,2025-12-13 23:45:00


## 5. Outliers & Domain-Specific Impossible Values

In [8]:
for name, coerced_df in coerced_datasets.items():
    ts_cols, _ = timestamp_config[name]
    value_cols = [c for c in coerced_df.columns if c not in ts_cols]

    stats_rows = []
    for col in value_cols:
        s = coerced_df[col].dropna()
        if s.empty:
            continue
        q01, q25, q50, q75, q99 = s.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
        iqr = q75 - q25
        std = s.std()

        stats_rows.append({
            "column": col,
            "min": round(float(s.min()), 2),
            "p01": round(float(q01), 2),
            "median": round(float(q50), 2),
            "mean": round(float(s.mean()), 2),
            "p99": round(float(q99), 2),
            "max": round(float(s.max()), 2),
            "iqr_1.5x": int(((s < q25 - 1.5 * iqr) | (s > q75 + 1.5 * iqr)).sum()),
            "iqr_3.0x": int(((s < q25 - 3.0 * iqr) | (s > q75 + 3.0 * iqr)).sum()),
            "z_gt_3": int((((s - s.mean()).abs() / std) > 3.0).sum()) if std > 0 else 0,
        })

    print(f"=== {name}: Distribution & Outlier Counts ===")
    display(pd.DataFrame(stats_rows))

    print(f"Domain-specific sanity checks ({name}):")
    if name == "Actual Consumption":
        for col in ["grid load [MWh]", "Grid load incl. hydro pumped storage [MWh]"]:
            print(f"  - {col} <= 0: {int((coerced_df[col] <= 0).sum()):,}")
        print(f"  - Hydro pumped storage [MWh] < 0: {int((coerced_df['Hydro pumped storage [MWh]'] < 0).sum()):,}")
    elif name == "Day-Ahead Prices":
        s = coerced_df[PRICE_COL].dropna()
        print(f"  - {PRICE_COL} outside SDAC caps (<-500 or >4000 €/MWh): {int(((s < -500) | (s > 4000)).sum()):,}")
    elif name == "Open-Meteo Weather":
        checks = [
            ("temperature_2m (°C)", lambda x: (x < -40) | (x > 50), "outside [-40, 50] °C"),
            ("dew_point_2m (°C)", lambda x: (x < -50) | (x > 45), "outside [-50, 45] °C"),
            ("relative_humidity_2m (%)", lambda x: (x < 0) | (x > 100), "outside [0, 100] %"),
            ("precipitation (mm)", lambda x: x < 0, "< 0 mm"),
            ("shortwave_radiation (W/m²)", lambda x: x < 0, "< 0 W/m²"),
            ("direct_radiation (W/m²)", lambda x: x < 0, "< 0 W/m²"),
            ("sunshine_duration (s)", lambda x: (x < 0) | (x > 900), "outside [0, 900] s"),
            ("wind_speed_10m (km/h)", lambda x: x < 0, "< 0 km/h"),
            ("weather_code (wmo code)", lambda x: (x < 0) | (x > 99), "outside [0, 99]"),
        ]
        for col, rule, desc in checks:
            s = coerced_df[col].dropna()
            print(f"  - {col} {desc}: {int(rule(s).sum()):,}")
        dew_gt_temp = int((coerced_df["dew_point_2m (°C)"] > coerced_df["temperature_2m (°C)"] + 1e-6).sum())
        print(f"  - dew_point_2m > temperature_2m (physically impossible): {dew_gt_temp:,}")
    print()

=== Actual Consumption: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x,z_gt_3
0,grid load [MWh],841.75,1012.25,1682.75,1708.12,2458.00,2723.50,0,0,0
1,Grid load incl. hydro pumped storage [MWh],879.30,1100.75,1805.75,1812.21,2512.75,2965.00,2,0,3
2,Hydro pumped storage [MWh],0.00,0.00,59.00,104.08,462.75,694.75,6205,19,3338
3,Residual load [MWh],-657.00,182.34,1422.75,1425.36,2339.75,2659.75,5374,63,2416


Domain-specific sanity checks (Actual Consumption):
  - grid load [MWh] <= 0: 0
  - Grid load incl. hydro pumped storage [MWh] <= 0: 0
  - Hydro pumped storage [MWh] < 0: 0

=== Day-Ahead Prices: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x,z_gt_3
0,Germany/Luxembourg [€/MWh],-500.00,-9.93,71.18,93.28,479.91,936.28,18913,8244,6549
1,∅ DE/LU neighbours [€/MWh],-152.18,0.30,79.26,99.32,453.92,819.36,15723,5908,5010
2,Belgium [€/MWh],-500.00,-5.48,59.55,85.52,448.87,871.00,27211,11824,7964
3,Denmark 1 [€/MWh],-440.10,-2.54,49.73,75.91,450.00,936.28,22304,10343,7868
4,Denmark 2 [€/MWh],-60.04,-0.60,49.04,75.44,449.99,936.31,23537,10822,7793
5,France [€/MWh],-134.94,-0.06,54.17,85.29,489.54,2987.78,31971,14808,8264
6,Netherlands [€/MWh],-500.00,-4.34,58.50,85.55,442.02,872.96,25070,10528,7847
7,Norway 2 [€/MWh],-61.84,0.88,45.50,66.87,417.62,898.25,28326,12399,7724
8,Austria [€/MWh],-500.00,-3.00,77.93,101.89,493.00,919.64,18137,7808,6605
9,Poland [€/MWh],-132.95,-0.01,92.37,99.68,318.11,771.00,8219,2077,3345


Domain-specific sanity checks (Day-Ahead Prices):
  - Germany/Luxembourg [€/MWh] outside SDAC caps (<-500 or >4000 €/MWh): 0

=== Open-Meteo Weather: Distribution & Outlier Counts ===


,column,min,p01,median,mean,p99,max,iqr_1.5x,iqr_3.0x,z_gt_3
0,temperature_2m (°C),-17.1,-4.6,11.0,11.28,29.3,36.1,190,0,150
1,relative_humidity_2m (%),14.0,32.0,77.0,73.51,99.0,100.0,393,0,232
2,precipitation (mm),-0.1,0.0,0.0,0.02,0.5,19.3,34631,34631,3169
3,shortwave_radiation (W/m²),0.0,0.0,2.0,140.33,809.0,930.0,24904,366,4350
4,direct_radiation (W/m²),0.0,0.0,0.0,84.09,671.0,806.0,48434,29313,9160
5,sunshine_duration (s),0.0,0.0,0.0,296.32,900.0,900.0,0,0,0
6,wind_speed_10m (km/h),0.0,1.0,7.8,8.91,26.6,52.0,9691,653,4139
7,weather_code (wmo code),0.0,0.0,3.0,10.47,80.0,99.0,47636,47636,7124
8,dew_point_2m (°C),-21.0,-8.9,6.4,6.19,18.0,23.2,889,0,847


Domain-specific sanity checks (Open-Meteo Weather):
  - temperature_2m (°C) outside [-40, 50] °C: 0
  - dew_point_2m (°C) outside [-50, 45] °C: 0
  - relative_humidity_2m (%) outside [0, 100] %: 0
  - precipitation (mm) < 0 mm: 1
  - shortwave_radiation (W/m²) < 0 W/m²: 0
  - direct_radiation (W/m²) < 0 W/m²: 0
  - sunshine_duration (s) outside [0, 900] s: 0
  - wind_speed_10m (km/h) < 0 km/h: 0
  - weather_code (wmo code) outside [0, 99]: 0
  - dew_point_2m > temperature_2m (physically impossible): 17



## 6. Generate & Save Baden-Württemberg (BW) Holiday Dataset

In [9]:
bw_holidays = holidays.Germany(years=range(2017, 2027), subdiv="BW")
df_holidays = (
    pd.DataFrame(list(bw_holidays.items()), columns=["Datum", "Feiertag"])
    .sort_values("Datum")
    .reset_index(drop=True)
)
df_holidays["Datum"] = pd.to_datetime(df_holidays["Datum"])
df_holidays.to_excel(HOLIDAY_FILE, index=False)

print(f"Saved {len(df_holidays)} BW holidays to: {HOLIDAY_FILE}")
display(df_holidays.head(10))

Saved 121 BW holidays to: C:\Users\Win11\Documents\GitHub\Energy-Forecasting\Dataset\raw\df_holiday.xlsx


,Datum,Feiertag
0,2017-01-01,Neujahr
1,2017-01-06,Heilige Drei Könige
2,2017-04-14,Karfreitag
3,2017-04-17,Ostermontag
4,2017-05-01,Erster Mai
5,2017-05-25,Christi Himmelfahrt
6,2017-06-05,Pfingstmontag
7,2017-06-15,Fronleichnam
8,2017-10-03,Tag der Deutschen Einheit
9,2017-10-31,Reformationstag


## 7. Checking duplicates (especially in timestamps), Align Timestamps & Merge Datasets
- **SMARD (`Consumption` & `Prices`)**: Local `Europe/Berlin` time disambiguated with `ambiguous='infer'` and converted to `UTC`.
- **Open-Meteo (`Weather`)**: Fixed `UTC+2` (`utc_offset_seconds = 7200`) localized and converted to `UTC`.
- **Holidays**: Matched against the local `Europe/Berlin` calendar date.

In [10]:
dup_summary_rows = []

for name, df in coerced_datasets.items():
    _, primary_ts = timestamp_config[name]
    ts = df[primary_ts]
    valid_ts = ts.dropna()
    full_dups = int(df.duplicated().sum())
    ts_dups = int(ts.duplicated().sum())

    dst_dup_rows = 0
    non_dst_dup_rows = 0
    if ts_dups > 0:
        dup_timestamps = valid_ts[valid_ts.duplicated(keep=False)]
        oct_dst_mask = (
            (dup_timestamps.dt.month == 10)
            & (dup_timestamps.dt.weekday == 6)
            & (dup_timestamps.dt.hour == 2)
        )
        dst_dup_rows = int(oct_dst_mask.sum())
        non_dst_dup_rows = int((~oct_dst_mask).sum())

    expected_grid = pd.date_range(start=valid_ts.min(), end=valid_ts.max(), freq="15min")
    missing_wall_clock = expected_grid.difference(valid_ts.unique())
    march_dst_mask = (
        (missing_wall_clock.month == 3)
        & (missing_wall_clock.weekday == 6)
        & (missing_wall_clock.hour == 2)
    )

    dup_summary_rows.append({
        "dataset": name,
        "start_ts": valid_ts.min(),
        "end_ts": valid_ts.max(),
        "full_row_duplicates": full_dups,
        "duplicate_timestamps": ts_dups,
        "oct_dst_fallback_rows": dst_dup_rows,
        "unexpected_duplicate_rows": non_dst_dup_rows,
        "march_dst_skipped_slots": int(march_dst_mask.sum()),
        "unexpected_missing_slots": int((~march_dst_mask).sum()),
    })

display(pd.DataFrame(dup_summary_rows))

,dataset,start_ts,end_ts,full_row_duplicates,duplicate_timestamps,oct_dst_fallback_rows,unexpected_duplicate_rows,march_dst_skipped_slots,unexpected_missing_slots
0,Actual Consumption,2017-01-01,2025-12-31 23:45:00,0,36,72,0,36,0
1,Day-Ahead Prices,2017-01-01,2025-12-31 23:45:00,0,36,72,0,36,0
2,Open-Meteo Weather,2017-01-01,2025-12-13 23:45:00,0,0,0,0,0,0


In [11]:
# 1. Consumption
cons_berlin = parse_timestamp_series(raw_consumption["Start date"]).dt.tz_localize(
    "Europe/Berlin", ambiguous="infer"
)
cons_df = pd.DataFrame({
    "timestamp_utc": cons_berlin.dt.tz_convert("UTC"),
    "timestamp_berlin": cons_berlin,
    TARGET_COL: pd.to_numeric(raw_consumption[TARGET_COL], errors="coerce"),
})

# 2. Day-Ahead Prices (repeat hourly value up to 4 quarter-hours)
prices_berlin = parse_timestamp_series(raw_prices["Start date"]).dt.tz_localize(
    "Europe/Berlin", ambiguous="infer"
)
price_series = pd.to_numeric(raw_prices[PRICE_COL], errors="coerce").ffill(limit=3)
prices_df = pd.DataFrame({
    "timestamp_utc": prices_berlin.dt.tz_convert("UTC"),
    PRICE_COL: price_series,
})

# 3. Open-Meteo Weather
fixed_utc_plus_2 = timezone(timedelta(hours=2))
weather_ts_utc = (
    parse_timestamp_series(raw_weather["time"])
    .dt.tz_localize(fixed_utc_plus_2)
    .dt.tz_convert("UTC")
)
weather_df = pd.DataFrame({"timestamp_utc": weather_ts_utc})
for col in WEATHER_COLS:
    weather_df[col] = pd.to_numeric(raw_weather[col], errors="coerce")

# 4. Merge on UTC timestamp & add BW holiday indicator
merged_df = (
    cons_df.merge(prices_df, on="timestamp_utc", how="inner")
    .merge(weather_df, on="timestamp_utc", how="inner")
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

holiday_dates = set(df_holidays["Datum"].dt.date)
merged_df["is_holiday"] = merged_df["timestamp_berlin"].dt.date.isin(holiday_dates).astype(int)

merged_df.to_csv(MERGED_FILE, index=False)
print(f"Saved merged dataset ({merged_df.shape[0]:,} rows x {merged_df.shape[1]} cols) to: {MERGED_FILE}")
print(f"UTC range:    {merged_df['timestamp_utc'].min()} -> {merged_df['timestamp_utc'].max()}")
print(f"Berlin range: {merged_df['timestamp_berlin'].min()} -> {merged_df['timestamp_berlin'].max()}")

display(merged_df.head())

Saved merged dataset (313,820 rows x 12 cols) to: C:\Users\Win11\Documents\GitHub\Energy-Forecasting\Dataset\preprocessed\merged_dataset.csv
UTC range:    2016-12-31 23:00:00+00:00 -> 2025-12-13 21:45:00+00:00
Berlin range: 2017-01-01 00:00:00+01:00 -> 2025-12-13 22:45:00+01:00


,timestamp_utc,timestamp_berlin,grid load [MWh],Germany/Luxembourg [€/MWh],temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),wind_speed_10m (km/h),dew_point_2m (°C),weather_code (wmo code),is_holiday
0,2016-12-31 23:00:00+00:00,2017-01-01 00:00:00+01:00,1544.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2016-12-31 23:15:00+00:00,2017-01-01 00:15:00+01:00,1529.50,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,1
2,2016-12-31 23:30:00+00:00,2017-01-01 00:30:00+01:00,1529.50,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,1
3,2016-12-31 23:45:00+00:00,2017-01-01 00:45:00+01:00,1530.75,NaN,NaN,NaN,0.0,0.0,NaN,NaN,0.0,1
4,2017-01-01 00:00:00+00:00,2017-01-01 01:00:00+01:00,1535.25,NaN,-5.7,88.0,0.0,0.0,4.2,-7.3,0.0,1


In [13]:
missing_summary = pd.DataFrame({
    "missing_count": merged_df.isna().sum(),
    "missing_%": (merged_df.isna().mean() * 100).round(2),
})
print("Missing values in merged dataset:")
display(missing_summary)

Missing values in merged dataset:


,missing_count,missing_%
timestamp_utc,0,0.00
timestamp_berlin,0,0.00
grid load [MWh],0,0.00
Germany/Luxembourg [€/MWh],61244,19.52
temperature_2m (°C),4,0.00
relative_humidity_2m (%),4,0.00
precipitation (mm),1,0.00
shortwave_radiation (W/m²),3,0.00
wind_speed_10m (km/h),4,0.00
dew_point_2m (°C),4,0.00


## 9. Target Variable Summary (`grid load [MWh]`)

In [ ]:
target = merged_df[TARGET_COL].dropna()
q01, q05, q25, q50, q75, q95, q99 = target.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

overall_stats = pd.DataFrame({
    "Metric": [
        "Count", "Missing", "Mean [MWh]", "Std [MWh]", "Min [MWh]",
        "1% Quantile [MWh]", "5% Quantile [MWh]", "25% Quantile [MWh]",
        "Median (50%) [MWh]", "75% Quantile [MWh]", "95% Quantile [MWh]",
        "99% Quantile [MWh]", "Max [MWh]", "IQR [MWh]", "Skewness", "Kurtosis",
    ],
    "Value": [
        f"{len(target):,}", f"{int(merged_df[TARGET_COL].isna().sum()):,}",
        f"{target.mean():.2f}", f"{target.std():.2f}", f"{target.min():.2f}",
        f"{q01:.2f}", f"{q05:.2f}", f"{q25:.2f}", f"{q50:.2f}", f"{q75:.2f}",
        f"{q95:.2f}", f"{q99:.2f}", f"{target.max():.2f}", f"{(q75 - q25):.2f}",
        f"{target.skew():.3f}", f"{target.kurtosis():.3f}",
    ],
})
print("Overall Target Distribution:")
display(overall_stats)

df_temp = merged_df[["timestamp_berlin", TARGET_COL, "is_holiday"]].dropna(subset=[TARGET_COL]).copy()
df_temp["year"] = df_temp["timestamp_berlin"].dt.year

print("Target Summary by Year:")
display(
    df_temp.groupby("year")[TARGET_COL]
    .agg(count="count", mean="mean", std="std", min="min", median="median", max="max")
    .round(2)
)


: 

## 10. Correlation Overview Between Variables

In [15]:
feature_cols = [TARGET_COL, PRICE_COL, *WEATHER_COLS, "is_holiday"]
corr_df = merged_df[feature_cols].dropna()
print(f"Correlation computed on {len(corr_df):,} complete 15-minute observations.")

pearson_corr = corr_df.corr(method="pearson")
spearman_corr = corr_df.corr(method="spearman")

target_corr = (
    pd.DataFrame({
        "Pearson_r": pearson_corr[TARGET_COL],
        "Spearman_rho": spearman_corr[TARGET_COL],
        "Abs_Pearson": pearson_corr[TARGET_COL].abs(),
    })
    .sort_values("Abs_Pearson", ascending=False)
    .drop(columns=["Abs_Pearson"])
    .round(4)
)

print(f"Correlation with Target ({TARGET_COL}):")
display(target_corr)

print("Full Pearson Correlation Matrix:")
display(pearson_corr.round(3))

Correlation computed on 252,576 complete 15-minute observations.
Correlation with Target (grid load [MWh]):


,Pearson_r,Spearman_rho
grid load [MWh],1.0000,1.0000
dew_point_2m (°C),-0.2554,-0.2508
is_holiday,-0.1989,-0.2041
shortwave_radiation (W/m²),0.1979,0.3365
temperature_2m (°C),-0.1864,-0.1964
Germany/Luxembourg [€/MWh],0.1734,0.2093
wind_speed_10m (km/h),0.1564,0.1443
relative_humidity_2m (%),-0.0529,-0.0730
weather_code (wmo code),0.0350,0.0727
precipitation (mm),0.0074,0.0453


Full Pearson Correlation Matrix:


,grid load [MWh],Germany/Luxembourg [€/MWh],temperature_2m (°C),relative_humidity_2m (%),precipitation (mm),shortwave_radiation (W/m²),wind_speed_10m (km/h),dew_point_2m (°C),weather_code (wmo code),is_holiday
grid load [MWh],1.000,0.173,-0.186,-0.053,0.007,0.198,0.156,-0.255,0.035,-0.199
Germany/Luxembourg [€/MWh],0.173,1.000,-0.010,0.056,-0.009,-0.111,-0.178,0.017,-0.006,-0.069
temperature_2m (°C),-0.186,-0.010,1.000,-0.570,0.005,0.520,-0.050,0.853,-0.082,-0.020
relative_humidity_2m (%),-0.053,0.056,-0.570,1.000,0.126,-0.640,-0.145,-0.068,0.279,0.008
precipitation (mm),0.007,-0.009,0.005,0.126,1.000,-0.064,0.098,0.081,0.429,-0.003
shortwave_radiation (W/m²),0.198,-0.111,0.520,-0.640,-0.064,1.000,0.042,0.217,-0.154,-0.004
wind_speed_10m (km/h),0.156,-0.178,-0.050,-0.145,0.098,0.042,1.000,-0.138,0.142,-0.010
dew_point_2m (°C),-0.255,0.017,0.853,-0.068,0.081,0.217,-0.138,1.000,0.066,-0.021
weather_code (wmo code),0.035,-0.006,-0.082,0.279,0.429,-0.154,0.142,0.066,1.000,-0.012
is_holiday,-0.199,-0.069,-0.020,0.008,-0.003,-0.004,-0.010,-0.021,-0.012,1.000
